# zscore-spy: pooled Z-score model, top 20, daily rebalance

The live version of the **Pooled top 20 (long)** strategy from `ai-stock-insights`. Out-of-sample backtest (Jan 2024 to Oct 2026, after 5 bps costs): 43.9% CAGR, beta 1.41, alpha 13.6%/yr with t = 1.05. Most of the return was market risk, so expect it to swing harder than SPY.

**Each day:**
1. Pull ~200 days of daily bars for the 366 stocks the backtest traded, plus SPY.
2. Build the 4 features (`z90`, `atr20_pct`, `vol_z90`, `spy_rsi14`) for each of the last 5 sessions.
3. The frozen pooled XGBoost (trained 1998-2020, `models/zscore_pooled.json`) predicts each stock's next 5-day return. Each session's top 20 is one tranche.
4. Target book = average of the last 5 tranches: 1% of equity per pick per day, up to 5% for a stock picked all 5 days. No stop-loss: a pick is held until its tranche ages out.
5. Send the difference to the OMS: sells first, wait for fills, then buys.

**When to run:** once per trading day, **between about 3:30 and 3:55 PM ET**. The backtest set weights at each close, and today's bar (15 minutes delayed) is the closest thing to that close. Outside market hours the notebook only shows the plan and sends nothing, because the sells must fill before the buys can use their cash.

Re-running on the same day is safe: order ids are fixed per day and symbol, so the OMS returns the earlier order instead of sending a second one.

In [1]:
# Settings
STRATEGY_ID = "zscore-spy"   # registered on the platform with $1,000
TOP_N = 20
HORIZON = 5                   # trading days each pick is held
MIN_TRADE = 1.00              # skip changes under $1 (Alpaca's fractional minimum)
INVEST_FRACTION = 0.99        # keep 1% in cash so notional buys never overshoot
DRY_RUN = False               # True prints the orders instead of sending them
FORCE_TRADE = False           # True sends orders even with the market closed (they queue for the open)

## 1. Connect to the OMS

In [2]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from alpaca.data.enums import Adjustment, DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient

from oms_client import OMS, alpaca_keys, open_buy_cost

oms = OMS(STRATEGY_ID, dry_run=DRY_RUN or None)
strategy = oms.strategy()
assert strategy is not None, f"{STRATEGY_ID} is not registered on the platform"
print(f"{STRATEGY_ID}: budget ${strategy['cash_budget']:,.2f}, spendable cash ${strategy['cash_remaining']:,.2f}")

clock = TradingClient(*alpaca_keys(), paper=True).get_clock()
print(f"Market {'OPEN' if clock.is_open else 'CLOSED'} | now {clock.timestamp:%Y-%m-%d %H:%M %Z} | next close {clock.next_close:%Y-%m-%d %H:%M}")

Started OMS backend at http://127.0.0.1:8787 (pid 74891, log in strategies/.logs/)


zscore-spy: budget $1,000.00, spendable cash $10.61
Market OPEN | now 2026-10-29 15:40 -04:00 | next close 2026-10-29 16:00


## 2. Model and universe

In [3]:
MODELS = Path("models")
meta = json.loads((MODELS / "zscore_universe.json").read_text())
FEATURES = meta["features"]
model = xgb.Booster()
model.load_model(MODELS / "zscore_pooled.json")

# The backtest used Yahoo tickers (BF-B); Alpaca uses BF.B.
TICKERS = [t.replace("-", ".") for t in meta["tickers"]]
print(f"{len(TICKERS)} tickers, features {FEATURES}, trained through {meta['trained_through']}")

366 tickers, features ['z90', 'atr20_pct', 'vol_z90', 'spy_rsi14'], trained through 2020-12-31


## 3. Daily bars

Full-market (SIP) bars with split and dividend adjustment, ending 16 minutes ago (the free data plan's delay). While the market is open, today's bar is still forming and stands in for today's close.

In [4]:
data = StockHistoricalDataClient(*alpaca_keys())
now = datetime.now(timezone.utc)
raw = data.get_stock_bars(StockBarsRequest(
    symbol_or_symbols=TICKERS + ["SPY"],
    timeframe=TimeFrame.Day,
    start=now - timedelta(days=220),
    end=now - timedelta(minutes=16),
    feed=DataFeed.SIP,
    adjustment=Adjustment.ALL,
)).df

bars = {sym: g.droplevel(0).set_index(g.index.get_level_values(1).tz_convert("America/New_York").normalize().tz_localize(None))
        for sym, g in raw.groupby(level=0)}
last_day = bars["SPY"].index[-1]
print(f"{len(bars)} symbols, last session {last_day:%Y-%m-%d}{' (in progress)' if clock.is_open else ''}")

367 symbols, last session 2026-10-29 (in progress)


## 4. Features and forecasts

Same formulas as `ai-stock-insights/insights/zscore.py`. Alpaca's adjusted close plays the role of Yahoo's `adj` column.

In [5]:
def rsi(close, n=14):
    d = close.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + up / dn)


def features(df, spy_rsi):
    px = df["close"]
    out = pd.DataFrame(index=df.index)
    out["z90"] = (px - px.rolling(90).mean()) / px.rolling(90).std()
    prev = px.shift()
    tr = pd.concat([df["high"] - df["low"], (df["high"] - prev).abs(), (df["low"] - prev).abs()], axis=1).max(axis=1)
    out["atr20_pct"] = tr.rolling(20).mean() / px * 100
    lv = np.log(df["volume"].where(df["volume"] > 0))
    out["vol_z90"] = (lv - lv.rolling(90).mean()) / lv.rolling(90).std()
    out["spy_rsi14"] = spy_rsi.reindex(df.index).ffill()
    return out


spy_rsi = rsi(bars["SPY"]["close"])
sessions = bars["SPY"].index[-HORIZON:]
rows = []
for t in TICKERS:
    if t in bars:
        f = features(bars[t], spy_rsi).reindex(sessions).dropna(subset=FEATURES)
        rows.append(f.assign(ticker=t))
feat = pd.concat(rows)
feat["pred"] = model.predict(xgb.DMatrix(feat[FEATURES]))
forecasts = feat.reset_index(names="date").pivot(index="date", columns="ticker", values="pred")
print(f"forecasts for {forecasts.notna().sum(axis=1).to_dict()}")

forecasts for {Timestamp('2026-10-23 00:00:00'): 366, Timestamp('2026-10-26 00:00:00'): 366, Timestamp('2026-10-27 00:00:00'): 366, Timestamp('2026-10-28 00:00:00'): 366, Timestamp('2026-10-29 00:00:00'): 366}


## 5. Target book: average of the last 5 daily top-20 tranches

In [6]:
picks = forecasts.rank(axis=1, ascending=False, method="first") <= TOP_N
tranches = picks.astype(float).div(picks.sum(axis=1), axis=0)
target = tranches.mean().loc[lambda w: w > 0].sort_values(ascending=False)

today = forecasts.iloc[-1].sort_values(ascending=False).head(TOP_N)
print(f"Today's top {TOP_N} ({forecasts.index[-1]:%Y-%m-%d}):")
display(pd.DataFrame({"forecast_5d_%": today.round(2),
                      "z90": feat[feat.index == forecasts.index[-1]].set_index("ticker").loc[today.index, "z90"].round(2)}).T)
print(f"Target book: {len(target)} names, weights {target.min():.1%} to {target.max():.1%}, sum {target.sum():.0%}")

Today's top 20 (2026-10-29):


ticker,MRVL,COHR,HPE,CIEN,TER,STX,LITE,SKYD,SMCI,FLEX,NOW,WDC,VST,INTC,TWLO,FIX,P,GLW,LRCX,FICO
forecast_5d_%,1.69,1.65,1.34,1.11,1.08,1.08,1.08,1.04,1.04,1.02,0.99,0.98,0.96,0.96,0.95,0.94,0.94,0.93,0.93,0.93
z90,-0.65,-1.45,0.51,-0.91,-2.02,-1.63,0.51,0.71,-0.07,-1.34,-0.94,-1.06,-0.90,-0.88,0.12,-2.13,1.31,-1.19,-1.56,-1.48


Target book: 47 names, weights 1.0% to 5.0%, sum 100%


## 6. Rebalance through the OMS

In [7]:
prices = {sym: float(df["close"].iloc[-1]) for sym, df in bars.items()}

if not clock.is_open and not FORCE_TRADE and not oms.dry_run:
    print("Market is closed: showing the plan only. Run again between 3:30 and 3:55 PM ET to trade.")
    oms.dry_run = True
    result = oms.rebalance(target.to_dict(), prices, INVEST_FRACTION, MIN_TRADE)
    oms.dry_run = False
else:
    result = oms.rebalance(target.to_dict(), prices, INVEST_FRACTION, MIN_TRADE)

equity $1,025.76 (cash $10.61) | 61 held, 47 targeted | 40 sells, 29 buys


sell ADM -> pending_new (order c6ac4786-e5fd-4254-b07c-392a8d1e941f, client id zscore-spy-20261029-s-ADM)


sell ADSK -> pending_new (order 290e8f8e-aa9a-4576-a598-6501c6b9f486, client id zscore-spy-20261029-s-ADSK)


sell AEE -> pending_new (order 07f19329-d2ee-4b7b-8429-acc42e0a5097, client id zscore-spy-20261029-s-AEE)


sell AEP -> pending_new (order b29eed75-918b-461a-8fd6-128461ed69c5, client id zscore-spy-20261029-s-AEP)


sell AKAM -> pending_new (order 7a14e0ba-dc58-4a93-897f-4c402bfe1d3b, client id zscore-spy-20261029-s-AKAM)


... (60 outputs left out)

buy TER -> pending_new (order 89ca4995-5702-4b99-aadc-cdee0cbb8474, client id zscore-spy-20261029-b-TER)


buy TWLO -> pending_new (order 9f35d87e-baee-458b-8dd3-2bbb7f4b7581, client id zscore-spy-20261029-b-TWLO)


buy VST -> pending_new (order 8b087dac-02b4-4561-b4be-5e0eb307cf1d, client id zscore-spy-20261029-b-VST)


buy WDC -> pending_new (order 90fa9e4f-962f-437b-8abf-a6d9ff17d775, client id zscore-spy-20261029-b-WDC)


## 7. Book after the rebalance

Buys fill a few seconds after they are sent. Re-run this cell to refresh.

In [8]:
import time
time.sleep(5)
book = oms.book()
s = book["strategy"]
pos = pd.DataFrame(book["positions"])
cash = s["cash_remaining"] + open_buy_cost(book["orders"])
if len(pos):
    pos["price"] = pos["symbol"].map(prices)
    pos["value"] = pos["qty"] * pos["price"]
    pos["weight"] = pos["value"] / (pos["value"].sum() + cash)
    display(pos[["symbol", "qty", "avg_price", "price", "value", "weight", "unrealized_pnl"]]
            .sort_values("value", ascending=False).round(4))
print(f"equity ${pos['value'].sum() if len(pos) else 0:,.2f} in positions + ${cash:,.2f} cash | "
      f"realized {s['realized_pnl']:+.2f} | unrealized {(s['unrealized_pnl'] or 0):+.2f}")

,symbol,qty,avg_price,price,value,weight,unrealized_pnl
33,LITE,0.0552,852.6492,920.9100,50.7951,0.0495,3.6962
36,FLEX,0.5437,81.6410,93.3750,50.7700,0.0495,6.3555
29,COHR,0.2141,224.2405,237.1100,50.7654,0.0495,2.7404
26,CIEN,0.1165,342.4051,348.6800,40.6312,0.0396,0.6520
41,INTC,0.4431,91.2281,91.6596,40.6112,0.0396,0.2135
8,STX,0.0459,647.4240,691.6600,31.7186,0.0309,2.0098
6,SKYD,4.0013,7.4200,7.8450,31.3906,0.0306,1.7006
7,SMCI,0.9088,32.6680,34.2450,31.1232,0.0303,1.4151
37,GLW,0.2366,127.8530,128.7300,30.4575,0.0297,0.1957
39,FIX,0.0230,1270.3798,1325.9200,30.4557,0.0297,1.2431


equity $862.04 in positions + $163.55 cash | realized +3.38 | unrealized +22.00
